# Bike Rental Prediction – Machine Learning Analysis

## Goals
This project analyzes a dataset of bike rentals and develops predictive models using:
- Exploratory Data Analysis (EDA)
- Ordinary Least Squares (OLS) regression on log-transformed target
- Ridge regression with hyperparameter tuning
- Cross-validation and learning curves
- A non-linear model (Random Forest or Neural Network)
- Final evaluation, discussion, limitations & future work

The objective is to create a clear, self-contained notebook demonstrating the full ML workflow.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV, learning_curve
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

sns.set(style="whitegrid")

---

# 2. Data Loading and Cleaning

In this section, we load the dataset, inspect its structure, and perform essential preprocessing such as:

- Checking data types  
- Handling missing values  
- Ensuring consistency  
- Converting and extracting information from the datetime column  

The goal is to prepare a clean dataset for analysis and modeling.


---

# 3. Exploratory Data Analysis (EDA)


The goal of this section is to develop an initial understanding of the dataset by examining its structure, main variables, and early patterns that may influence model design.

---

### 3.1 Dataset Overview

We begin by loading the training dataset and inspecting its structure.  
The dataset contains hourly bicycle counts recorded at several counting stations across Paris. Each row represents the number of bicycles detected at a specific counter (`counter_id`) at a given timestamp.

In many bike-sharing and traffic forecasting problems, it is common to use a log-transformed target variable:

\[
\text{log\_bike\_count} = \ln(\text{bike\_count} + 1)
\]

This transformation is useful because:

1. **Reducing skewness:** Bike traffic has many low-count hours and a few very high-count periods.  
2. **Stabilizing variance:** Helps linear models capture relationships more effectively.  
3. **Interpretability:** Changes in log-scale correspond to percentage-level changes in demand.

Whether or not we ultimately use the log-transformation will be evaluated during the modeling phase.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# sns.set_theme()

data = pd.read_parquet(Path("data") / "train.parquet")
data.head()


We now inspect the structure of the dataset:

- `data.info()` provides the data types and missing-value overview.  
- `data.nunique()` shows how many unique values exist in key fields such as counters and timestamps.

In [ ]:
data.info()
data.nunique(axis=0)


<span style="color:darkblue">

The dataset contains **56 counters** grouped into **30 sites**, meaning several counters are colocated at the same physical location.  
Coordinates and site identifiers confirm this structure.

There are **8,974 unique timestamps**, indicating broad temporal coverage.  
`bike_count` and `log_bike_count` show similar variability, each with nearly 1,000 distinct values.

</span>


#### Distribution of Activity Across Counters

Before analyzing temporal or weather-related patterns, it is useful to understand how bicycle traffic is distributed across counting stations.

Some counters record substantially higher traffic than others.  
High-volume counters typically correspond to major commuting corridors or central areas, while low-volume counters might be located in residential zones or less frequently used routes.

The following table lists the counters with the highest total recorded bike counts.

In [ ]:
(
    data.groupby("counter_id")["bike_count"]
        .sum()
        .sort_values(ascending=False)
        .head(10)
        .to_frame(name="total_bike_count")
)


<span style="color:darkblue">

The busiest counters each record well over **1 million bicycles**, indicating major commuting corridors.  
Many appear in **paired IDs**, suggesting counters measuring opposite directions at the same site.

Traffic volume varies strongly across locations, reflecting substantial spatial heterogeneity in bike usage.

</span>


### 3.2 Visualizing the Data

#### 3.2.1. Spatial Distribution of Counters

We first visualize the spatial distribution of all counting stations across Paris.
Using the geographic coordinates (`latitude`, `longitude`), we place a marker for each counter on an interactive map.

This helps verify that locations are correctly recorded and provides an initial sense of where bicycle traffic is being measured in the city.

In [ ]:
import folium

# Center the map on the mean latitude and longitude of all counters
m = folium.Map(location=data[["latitude", "longitude"]].mean(axis=0), zoom_start=13)

# Add one marker per counter
for _, row in (
    data[["counter_name", "latitude", "longitude"]]
    .drop_duplicates("counter_name")
    .iterrows()
):
    folium.Marker(
        location=row[["latitude", "longitude"]].values.tolist(),
        popup=row["counter_name"],
    ).add_to(m)

m


#### 3.2.2. Temporal Patterns at a Single Counter

To gain a more detailed understanding of temporal dynamics, we focus on one representative counting station.
By inspecting its time series over the full observation period, we can visually assess long-term trends, seasonality, and noise in the bike counts.


In [ ]:
# Select a specific counter (example: a busy central counter)
mask = data["counter_name"] == "Totem 73 boulevard de Sébastopol S-N"

# Filter and aggregate (here aggregation is trivial if each timestamp appears once)
data_mask = data[mask].copy()
data_mask["date"] = pd.to_datetime(data_mask["date"])  # Ensure datetime type
data_agg = data_mask.groupby("date", as_index=False)["bike_count"].sum()

# Plot bike counts over time for this counter
data_agg.plot(x="date", y="bike_count", title="Bike Count Over Time", legend=True)


We next aggregate the time series at a weekly level for the same counter.
This smooths out hourly fluctuations and highlights broader patterns such as seasonal trends or sustained growth/decline in bike usage.

In [ ]:
mask = data["counter_name"] == "Totem 73 boulevard de Sébastopol S-N"

(
    data[mask]
    .groupby(pd.Grouper(freq="1w", key="date"))[["bike_count"]]
    .sum()
    .plot(title="Weekly Aggregated Bike Count")
)


<span style="color:darkblue">

Weekly totals show a clear **seasonal pattern**: bike usage drops sharply in winter months (December–February) and rises again approaching late spring and summer.  
This reflects typical weather-driven cycling behaviour, with warmer periods supporting higher mobility across the city.

</span>

#### 3.2.3. Zooming into a Single Week

To better understand intra-week patterns, we zoom into one specific week for the same counter.
This allows us to compare workdays and weekends in terms of hourly or daily traffic.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# Filter for the specific counter and a chosen date range
mask = (
    (data["counter_name"] == "Totem 73 boulevard de Sébastopol S-N")
    & (data["date"] > pd.to_datetime("2021-03-01"))
    & (data["date"] < pd.to_datetime("2021-03-08"))
)

data_filtered = data[mask].copy()

# Optionally aggregate (depending on time resolution)
data_filtered = data_filtered.groupby("date", as_index=False)["bike_count"].sum()

# Plot the filtered data
data_filtered.plot(x="date", y="bike_count", ax=ax, marker=".", legend=False)
ax.set_title("Bike Count from March 1 to March 8, 2021")
ax.set_ylabel("Bike Count")
ax.set_xlabel("Date")
plt.show()


<span style="color:darkblue">

Within a single week, strong **weekday rush-hour cycles** appear with two clear peaks per day (morning and evening commutes). On weekends, like on 6th and 7th March, the shape of the during the day looks different. In terms of daily peaks, there seem to be slight differences when comparing Monday-Wednesday with Thursday & Friday.

</span>

#### 3.2.4. Calendar Feature Encoding

Bike usage is strongly driven by time-related effects such as hour of the day, day of the week, and season.
To capture these effects, we decompose the `date` variable into several calendar components:

- `year`
- `month`
- `day`
- `weekday` (0 = Monday, ..., 6 = Sunday)
- `hour`

These features will later be used as predictors in our regression models.


In [ ]:
def _encode_dates(X: pd.DataFrame) -> pd.DataFrame:
    """Add calendar features derived from the 'date' column."""
    X = X.copy()
    X["date"] = pd.to_datetime(X["date"])
    X["year"] = X["date"].dt.year
    X["month"] = X["date"].dt.month
    X["day"] = X["date"].dt.day
    X["weekday"] = X["date"].dt.weekday  # 0=Monday, 6=Sunday
    X["hour"] = X["date"].dt.hour
    return X

data = _encode_dates(data)


#### 3.2.5. Aggregate Weekday Patterns

We now aggregate the total bike counts by weekday, summing over all counters and all timestamps.
This provides an overall view of how bicycle usage varies from Monday to Sunday across the entire network.


In [ ]:
# Aggregate bike counts by weekday (0 = Monday, ..., 6 = Sunday)
weekday_aggregates = data.groupby("weekday")["bike_count"].sum()

# Define order and labels
weekday_order = [0, 1, 2, 3, 4, 5, 6]
weekday_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

weekday_aggregates = weekday_aggregates.reindex(weekday_order)

plt.figure(figsize=(10, 6))
weekday_aggregates.index = weekday_names
weekday_aggregates.plot(kind="bar", edgecolor="black")
plt.title("Total Bike Count by Weekday")
plt.ylabel("Total Bike Count")
plt.xlabel("Day of the Week")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


<span style="color:darkblue">

Aggregated across all counters, **Tuesday to Thursday** show the highest bike usage, while Monday and Friday are slightly lower. Weekend days exhibit a pronounced reduction in total counts, consistent with fewer commuting trips and more variable recreational activity.

</span>

#### 3.2.6. Weekday Patterns by Counter (Normalized to Monday)

To examine whether weekday patterns are consistent across locations, we compute the total bike count per weekday for each counter and normalize these values by the respective Monday count.

This yields a percentage index where Monday corresponds to 100%, and other days indicate relative increases or decreases compared to Monday for each station.


In [ ]:
# Sum bike counts by station and weekday
station_weekday_counts = data.groupby(["counter_name", "weekday"])["bike_count"].sum().unstack()

# Normalize by Monday (weekday = 0) for each station
normalized_counts = station_weekday_counts.div(station_weekday_counts[0], axis=0) * 100

# Plot normalized weekday distributions for all stations
plt.figure(figsize=(12, 6))
for station in normalized_counts.index:
    plt.plot(normalized_counts.columns, normalized_counts.loc[station], alpha=0.6)

plt.title("Normalized Bike Counts by Weekday (Percentage of Monday) for Each Counter")
plt.ylabel("Percentage of Monday's Count (%)")
plt.xlabel("Weekday (0=Monday, ..., 6=Sunday)")
plt.tight_layout()
plt.show()


<span style="color:darkblue">

Across stations, weekday profiles are remarkably consistent: counts remain close to **110–120% of Monday levels** from Tuesday to Thursday. In contrast, Saturday and Sunday display far greater variation between counters, suggesting location-specific differences in recreational cycling behaviour.

Maybe it would makes sense to group Tuesday until Thursday as one category, given they are all at an equal level in terms of counts.

</span>

#### 3.2.7. Distribution of the Target Variable

Understanding the distribution of the target variable is important before fitting models.
Least-squares–based regression methods (such as OLS or Ridge) assume errors that are approximately normally distributed, which is often easier to satisfy when the target itself is closer to a symmetric distribution.

We therefore begin by inspecting the raw distribution of `bike_count`.


In [ ]:
ax = sns.histplot(data, x="bike_count", kde=True, bins=50)
plt.title("Distribution of Raw Bike Counts")
plt.show()


<span style="color:darkblue">

The distribution of `bike_count` is extremely right-skewed: most observations correspond to very low bicycle flows, while a small number of hours show exceptionally high usage. This long-tailed behaviour is typical of mobility datasets and suggests that a direct modeling of `bike_count` may violate normal-error assumptions.

</span>

Because of the strong skewness, applying a log-transformation can help stabilize variance  
and make the distribution more symmetric. We now examine the distribution of the transformed  
variable `log_bike_count`, defined as:

$$
\text{log\_bike\_count} = \ln(\text{bike\_count} + 1)
$$


In [ ]:
ax = sns.histplot(data, x="log_bike_count", kde=True, bins=50)
plt.title("Distribution of Log-Transformed Bike Count")
plt.show()

<span style="color:darkblue">

The log-transformation substantially reduces skewness, producing a more balanced and bell-shaped distribution.  
Although still not perfectly Gaussian, the transformed variable aligns more closely with the assumptions of linear models and is therefore a promising candidate for the modeling phase.

</span>

### 3.3 Correlation Analysis

To better understand the relationships between the target variable and the time-related features, we compute pairwise Pearson correlations between:

- `log_bike_count` (transformed target),
- the original `bike_count`,
- and the calendar features `hour`, `weekday`, `month`, and `year`.

Visualising these correlations in a heatmap helps identify which variables appear most informative for explaining variation in bicycle usage.


In [ ]:
# Select relevant features for correlation analysis
correlation_features = ["log_bike_count", "hour", "weekday", "month", "year", "bike_count"]

# Compute the correlation matrix
correlation_matrix = data[correlation_features].corr()

# Plot the correlation heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Correlation Heatmap with Log Bike Count")
plt.show()


<span style="color:darkblue">

`log_bike_count` shows its strongest association with the **original bike count**, as expected, and a **moderate positive correlation with the hour of the day**, reflecting the strong daily cycle in bike usage.  
Correlations with `weekday`, `month`, and `year` are comparatively weak at this aggregated level, indicating that **short-term temporal structure (hourly effects)** dominates over coarser calendar effects.

</span>

### 3.4 Time-of-Day Phases

The correlation analysis suggests that the hour of the day is an important driver of bike usage.  
As an exploratory step, we group hours into broader **time-of-day phases** (Night, Morning, Midday, Afternoon, Evening, Late Evening) and study how the average `log_bike_count` behaves across these categories.

This can be seen as a simple form of feature engineering that replaces the raw hour with a coarser, interpretable time-of-day variable.

In [ ]:
def _add_time_phases(X: pd.DataFrame) -> pd.DataFrame:
    """Add a categorical 'time_of_day' feature based on the hour of day."""
    X = X.copy()
    # Create the 'time_of_day' categorical variable
    X["time_of_day"] = pd.cut(
        X["hour"],
        bins=[0, 5, 9, 13, 17, 20, 23],
        labels=["Night", "Morning", "Midday", "Afternoon", "Evening", "Late Evening"],
        right=True,
    )
    return X

# Apply the function to the dataset
data = _add_time_phases(data)

# Quick preview
data[["hour", "time_of_day"]].head()


We now compare the mean `log_bike_count` computed:

- for each individual hour of the day, and  
- for each time-of-day phase.

This allows us to assess whether aggregating hours into broad phases preserves or hides important hourly structure.

In [ ]:
# Mean and median by time-of-day phase
time_of_day_stats = data.groupby("time_of_day")["log_bike_count"].agg(["mean", "median"]).reset_index()

# Mean by hour of day
hourly_stats = data.groupby("hour")["log_bike_count"].mean().reset_index()

# Two subplots: hourly mean and phase mean
fig, ax = plt.subplots(2, 1, figsize=(8, 8), sharey=True)

# Plot hourly mean
ax[0].plot(hourly_stats["hour"], hourly_stats["log_bike_count"], label="Hourly Mean", marker="o")
ax[0].set_title("Hourly Mean of Log Bike Count")
ax[0].set_ylabel("Log Bike Count")
ax[0].set_xlabel("Hour of the Day")
ax[0].grid(True)

# Plot time-of-day mean as bar chart
time_of_day_stats.plot(
    x="time_of_day",
    y="mean",
    kind="bar",
    ax=ax[1],
    color="red",
    legend=False,
    edgecolor="black",
)
ax[1].set_title("Time of Day Mean of Log Bike Count")
ax[1].set_ylabel("Log Bike Count")
ax[1].set_xlabel("Time of Day")
ax[1].set_xticks(range(len(time_of_day_stats["time_of_day"])))
ax[1].set_xticklabels(time_of_day_stats["time_of_day"], rotation=45)

plt.tight_layout()
plt.show()


<span style="color:darkblue">

The hourly mean plot reveals a detailed **double-peak structure** with sharp morning and evening rush hours, and lower activity at night.  
When hours are aggregated into coarse time-of-day phases, these nuances largely disappear: phase means smooth over intra-phase variability and fail to fully capture the commuting peaks.

This suggests that, for modeling purposes, retaining the **original hourly resolution** (or using more refined encodings such as cyclical transforms) is preferable to replacing it with broad time-of-day categories.

</span>

---

# 4. Feature Engineering

To improve predictive performance, we engineer additional features:

### Time-based features
- Hour of day  
- Weekday vs weekend  
- Month / season  

### Weather transformations
- Non-linear terms (e.g., temperature²)  
- Binary indicators (e.g., extreme weather)

### Optional future extensions
- Lag features (previous hour/day rentals)  
- Rolling averages  

These engineered features form the input for the machine learning models.

---

# 5. Baseline Model — Ordinary Least Squares (OLS)

We train a simple Linear Regression model using:

- Log-transformed rental counts (`log1p(count)`)
- Feature set engineered in Section 4  
- Evaluation on test data using RMSE and R²  

This acts as the baseline for comparing more advanced models.

### Interpretation
We analyze regression coefficients to understand which features increase or decrease expected rental demand.

---

# 6. Regularized Model — Ridge Regression

Ridge Regression introduces L2 regularization to reduce overfitting in the linear model.

### Steps
- Standardize features using a pipeline  
- Tune the `alpha` parameter using GridSearchCV  
- Evaluate performance on the test set  
- Compare results with OLS  

Ridge often improves generalization and stabilizes coefficients.

---

# 7. Model Evaluation: Cross-Validation and Learning Curves

We evaluate the robustness and generalization of the models using:

### Cross-validation
- Compute cross-validated RMSE  
- Compare OLS and Ridge  

### Learning curves
Plot training and validation error as a function of training data size, allowing us to identify:

- High bias (underfitting)  
- High variance (overfitting)  
- Whether more data would help 


---

# 8. Advanced Model — Random Forest

We train a non-linear ensemble model capable of capturing complex interactions.

### Included steps
- Fit a Random Forest Regressor  
- Evaluate RMSE and R²  
- Analyze feature importances  

The Random Forest often provides strong predictive performance and helps reveal which features are truly influential.










 





---

# 9. Results and Discussion

In this section, we summarize and interpret the outcomes:

### Performance comparison
- OLS  
- Ridge Regression  
- Random Forest  

### Error patterns
Residual analysis to diagnose where models perform poorly (e.g., peak hours, extreme weather).

### Interpretation
Discuss why certain models performed better and what key factors drive bike rental demand.




---

# 10. Limitations and Future Work

### Limitations
- Dataset may not include all relevant drivers (events, holidays, bike availability).  
- Log-transform introduces mild bias when converting predictions back.  
- Standard regression models do not explicitly model temporal dependencies.  
- Random Forest lacks interpretability compared to linear models.  

### Future Work
- Add lagged features and rolling windows to capture temporal structure.  
- Explore time-series models (Prophet, ARIMA, LSTM).  
- Include richer weather and event-related datasets.  
- Investigate probabilistic forecasting for uncertainty quantification.  

---